# Diagnóstico inicial de calidad

## Resumen

Este notebook evalúa el CSV crudo sin modificarlo. Revisa estructura, tipos, faltantes semánticos, cardinalidad, duplicados, dominios, formatos y relaciones entre variables. Los resultados constituyen la línea base del informe Antes/Después.

## Configuración y datos

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src import catalogos, diagnostico, validadores

In [ ]:
RUTA_RAW = ROOT / "data" / "raw" / "establecimientos_raw.csv"
df = pd.read_csv(RUTA_RAW, dtype="string", keep_default_na=False)
print(f"Registros: {df.shape[0]:,} | Variables: {df.shape[1]}")

Registros: 11,891 | Variables: 17


## 1. Registros, variables y tipos

In [ ]:
diagnostico.resumen_tipos(df)

                tipo_observado        tipo_esperado
columna                                            
CODIGO                  string  texto identificador
DISTRITO                string  texto identificador
DEPARTAMENTO            string     texto categórico
MUNICIPIO               string     texto categórico
ESTABLECIMIENTO         string                texto
DIRECCION               string                texto
TELEFONO                string  texto identificador
SUPERVISOR              string                texto
DIRECTOR                string                texto
NIVEL                   string     texto categórico
SECTOR                  string     texto categórico
AREA                    string     texto categórico
STATUS                  string     texto categórico
MODALIDAD               string     texto categórico
JORNADA                 string     texto categórico
PLAN                    string     texto categórico
DEPARTAMENTAL           string     texto categórico

Los identificadores, teléfonos, categorías y textos deben conservarse como texto. Leerlos como números eliminaría ceros iniciales y no aportaría operaciones numéricas válidas.

## 2. Faltantes por variable y total

In [ ]:
diagnostico.resumen_faltantes(df)

                 faltantes  porcentaje
DIRECTOR              2165       18.21
TELEFONO               969        8.15
SUPERVISOR             562        4.73
DISTRITO               556        4.68
DIRECCION              112        0.94
ESTABLECIMIENTO         28        0.24
CODIGO                  23        0.19
MUNICIPIO               23        0.19
DEPARTAMENTO            23        0.19
NIVEL                   23        0.19
SECTOR                  23        0.19
AREA                    23        0.19
STATUS                  23        0.19
MODALIDAD               23        0.19
JORNADA                 23        0.19
PLAN                    23        0.19
DEPARTAMENTAL           23        0.19

In [ ]:
diagnostico.resumen_faltantes_total(df)

celdas_faltantes             4645.0
celdas_totales             202147.0
porcentaje                      2.3
variables_con_faltantes        17.0
dtype: float64

El conteo incluye `NA`, cadenas vacías, puntos, `SIN DATO` y cadenas formadas únicamente por guiones. Esto evita subestimar especialmente los faltantes de `DIRECTOR`.

## 3. Valores únicos

In [ ]:
diagnostico.contar_unicos(df)

CODIGO             11868
DISTRITO            1681
DEPARTAMENTO          23
MUNICIPIO            352
ESTABLECIMIENTO     6312
DIRECCION           7432
TELEFONO            6572
SUPERVISOR          1279
DIRECTOR            5490
NIVEL                  1
SECTOR                 4
AREA                   3
STATUS                 5
MODALIDAD              2
JORNADA                6
PLAN                  13
DEPARTAMENTAL         26
Name: valores_unicos, dtype: int64

## 4. Duplicados exactos y filas estructurales

In [ ]:
filas_totalmente_vacias = df.apply(diagnostico.mascara_faltantes).all(axis=1)
print("Filas totalmente vacías:", int(filas_totalmente_vacias.sum()))
print("Duplicados exactos:", diagnostico.contar_duplicados_exactos(df))
print(
    "Duplicados exactos excluyendo filas vacías:",
    diagnostico.contar_duplicados_exactos(df.loc[~filas_totalmente_vacias]),
)

Filas totalmente vacías: 23
Duplicados exactos: 22
Duplicados exactos excluyendo filas vacías: 0


## 5. Dominios geográficos

In [ ]:
departamentos_invalidos = df.loc[
    df["DEPARTAMENTO"].str.strip().ne("")
    & ~df["DEPARTAMENTO"].map(catalogos.es_departamento_valido),
    "DEPARTAMENTO",
].value_counts()
print("Departamento fuera del catálogo:")
departamentos_invalidos

Departamento fuera del catálogo:


DEPARTAMENTO
CIUDAD CAPITAL    2161
Name: count, dtype: Int64

In [ ]:
filas_geograficas = df[
    df["DEPARTAMENTO"].str.strip().ne("") & df["MUNICIPIO"].str.strip().ne("")
].copy()
municipio_valido = filas_geograficas.apply(
    lambda fila: catalogos.es_municipio_valido(
        fila["MUNICIPIO"], fila["DEPARTAMENTO"]
    ),
    axis=1,
)
municipios_fuera_dominio = filas_geograficas.loc[~municipio_valido, "MUNICIPIO"]
print("Filas fuera del catálogo antes de equivalencias:", int((~municipio_valido).sum()))
municipios_fuera_dominio.value_counts().head(30)

Filas fuera del catálogo antes de equivalencias: 2327


MUNICIPIO
ZONA 1                          868
ZONA 7                          236
ZONA 12                         149
ZONA 18                         134
ZONA 2                          118
ZONA 6                           94
ZONA 11                          80
ZONA 19                          65
ZONA 3                           61
ZONA 13                          61
ZONA 10                          56
ZONA 5                           45
ZONA 21                          40
ZONA 9                           38
ZONA 17                          30
SAN MIGUEL USPANTAN              26
LA TINTA                         23
ZONA 14                          22
SANTO TOMAS CHICHICASTENANGO     21
ZONA 16                          20
ZONA 15                          20
PACHALUN                         18
SAN PEDRO YEPOCAPA               17
COLOMBA COSTA CUCA               17
GENOVA COSTA CUCA                11
ZONA 8                           10
SAN MIGUEL TUCURU                 8
SAN JOSE EL RODEO 

Las zonas capitalinas no son municipios. Los municipios oficiales `SIPACATE`, `LA LIBERTAD`, `SANTA BARBARA` y `PETATAN` sí están incluidos en el catálogo corregido y ya no se cuentan como inválidos.

## 6. Formatos de texto e identificadores

In [ ]:
diagnostico.resumen_formato_texto(df)

                 espacios_extremos  espacios_multiples  caracteres_invisibles  \
variable                                                                        
CODIGO                           0                   0                      0   
DISTRITO                         0                   0                      0   
DEPARTAMENTO                     0                   0                      0   
MUNICIPIO                        0                   0                      0   
ESTABLECIMIENTO                  0                   0                      0   
DIRECCION                        0                   0                      0   
TELEFONO                         0                   0                      0   
SUPERVISOR                       0                   0                      0   
DIRECTOR                         0                   0                      0   
NIVEL                            0                   0                      0   
SECTOR                      

In [ ]:
diagnostico.resumen_patrones_identificadores(df)

                       patron  validos  invalidos_no_faltantes
variable                                                      
CODIGO          NN-NN-NNNN-NN    11868                       0
DISTRITO  NN-NNN o NN-NN-NNNN    11265                      70

In [ ]:
telefonos = df["TELEFONO"].mask(diagnostico.mascara_faltantes(df["TELEFONO"]))
telefonos_no_estandar = telefonos.notna() & ~telefonos.str.fullmatch(r"[2-7]\d{7}", na=False)
print("Teléfonos no vacíos fuera del formato único de 8 dígitos:", int(telefonos_no_estandar.sum()))
telefonos[telefonos_no_estandar].value_counts().head(20)

Teléfonos no vacíos fuera del formato único de 8 dígitos: 258


TELEFONO
24328801-24329098             4
79486433-79480166             4
2325732, 2320075, 2307014     3
78739432-79649696             3
79540830-79540909             3
4431414, 3109992              3
77648506-45419234-41177068    3
00000000                      3
78393245-78393246             2
23267252-09442325             2
24314989, 24310955            2
22067                         2
66609219, 22305978            2
8441160, 8441331              2
586577                        2
7844007                       2
78887460-55022177             2
78881148-78891068             2
24493119-24493733             2
5944312-5986774               2
Name: count, dtype: Int64

## 7. Consistencia entre variables

In [ ]:
comparables = df[
    df["DEPARTAMENTO"].str.strip().ne("")
    & df["DEPARTAMENTAL"].str.strip().ne("")
].copy()
depto_para_regla = comparables["DEPARTAMENTO"].replace("CIUDAD CAPITAL", "GUATEMALA")
relacion_valida = [
    validadores.es_departamental_consistente(depto, direccion)
    for depto, direccion in zip(depto_para_regla, comparables["DEPARTAMENTAL"])
]
print("Diferencias literales DEPARTAMENTO/DEPARTAMENTAL:", int(
    comparables["DEPARTAMENTO"].ne(comparables["DEPARTAMENTAL"]).sum()
))
print("Relaciones administrativas no permitidas:", int((~pd.Series(relacion_valida)).sum()))

Diferencias literales DEPARTAMENTO/DEPARTAMENTAL: 6095
Relaciones administrativas no permitidas: 0


Una diferencia literal no implica contradicción: `GUATEMALA NORTE` o `QUICHE NORTE` son direcciones departamentales válidas. La limpieza normaliza tildes y conserva estas subregiones.

## 8. Resumen de problemas para las 17 variables

In [ ]:
resumen_problemas = pd.DataFrame(
    [
        ("CODIGO", "Formato válido en registros reales; faltantes solo en filas vacías."),
        ("DISTRITO", "Faltantes y valores incompletos; coexisten dos formatos completos."),
        ("DEPARTAMENTO", "CIUDAD CAPITAL está fuera del catálogo de 22 departamentos."),
        ("MUNICIPIO", "Zonas capitalinas y variantes de nombres; validar contra 340 municipios."),
        ("ESTABLECIMIENTO", "Faltantes residuales y variantes por tildes; texto libre."),
        ("DIRECCION", "Faltantes, puntos como ausencia y redacción libre."),
        ("TELEFONO", "Faltantes, múltiples números, letras y longitudes inconsistentes."),
        ("SUPERVISOR", "Faltantes y variantes ortográficas; texto libre."),
        ("DIRECTOR", "Faltantes subestimados por guiones, puntos y SIN DATO."),
        ("NIVEL", "Un único valor real: DIVERSIFICADO; faltantes estructurales."),
        ("SECTOR", "Cuatro categorías válidas; faltantes estructurales."),
        ("AREA", "Tres categorías; SIN ESPECIFICAR es valor explícito."),
        ("STATUS", "Cinco estados válidos con significados diferentes."),
        ("MODALIDAD", "Dos categorías; normalizar diferencias de acentuación."),
        ("JORNADA", "Seis categorías; SIN JORNADA es categoría, no NA."),
        ("PLAN", "Trece categorías; las modalidades semipresenciales no son equivalentes."),
        ("DEPARTAMENTAL", "26 direcciones, incluidas subregiones y diferencias de tildes."),
    ],
    columns=["Variable", "Problema potencial de calidad"],
)
resumen_problemas

           Variable                      Problema potencial de calidad
0            CODIGO  Formato válido en registros reales; faltantes ...
1          DISTRITO  Faltantes y valores incompletos; coexisten dos...
2      DEPARTAMENTO  CIUDAD CAPITAL está fuera del catálogo de 22 d...
3         MUNICIPIO  Zonas capitalinas y variantes de nombres; vali...
4   ESTABLECIMIENTO  Faltantes residuales y variantes por tildes; t...
5         DIRECCION  Faltantes, puntos como ausencia y redacción li...
6          TELEFONO  Faltantes, múltiples números, letras y longitu...
7        SUPERVISOR   Faltantes y variantes ortográficas; texto libre.
8          DIRECTOR  Faltantes subestimados por guiones, puntos y S...
9             NIVEL  Un único valor real: DIVERSIFICADO; faltantes ...
10           SECTOR  Cuatro categorías válidas; faltantes estructur...
11             AREA  Tres categorías; SIN ESPECIFICAR es valor expl...
12           STATUS  Cinco estados válidos con significados diferen...
13    

## Conclusión

El CSV crudo es utilizable como fuente de limpieza, pero no debe analizarse directamente: contiene filas estructurales, faltantes codificados de varias maneras, formatos telefónicos/distritales inconsistentes y geografía que requiere equivalencias explícitas.